In [4]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [5]:
import os

In [6]:
### 数据预处理

In [7]:
base_dir = './dataset'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')


In [8]:
train_dir

'./dataset\\train'

In [9]:
test_dir

'./dataset\\test'

In [10]:
filenames = os.listdir(base_dir)

In [11]:
filenames

['Advertising.csv',
 'cloudy1.jpg',
 'cloudy10.jpg',
 'cloudy100.jpg',
 'cloudy101.jpg',
 'cloudy102.jpg',
 'cloudy103.jpg',
 'cloudy104.jpg',
 'cloudy105.jpg',
 'cloudy106.jpg',
 'cloudy107.jpg',
 'cloudy108.jpg',
 'cloudy109.jpg',
 'cloudy11.jpg',
 'cloudy110.jpg',
 'cloudy111.jpg',
 'cloudy112.jpg',
 'cloudy113.jpg',
 'cloudy114.jpg',
 'cloudy115.jpg',
 'cloudy116.jpg',
 'cloudy117.jpg',
 'cloudy118.jpg',
 'cloudy119.jpg',
 'cloudy12.jpg',
 'cloudy120.jpg',
 'cloudy121.jpg',
 'cloudy122.jpg',
 'cloudy123.jpg',
 'cloudy124.jpg',
 'cloudy125.jpg',
 'cloudy126.jpg',
 'cloudy127.jpg',
 'cloudy128.jpg',
 'cloudy129.jpg',
 'cloudy13.jpg',
 'cloudy130.jpg',
 'cloudy131.jpg',
 'cloudy132.jpg',
 'cloudy133.jpg',
 'cloudy134.jpg',
 'cloudy135.jpg',
 'cloudy136.jpg',
 'cloudy137.jpg',
 'cloudy138.jpg',
 'cloudy139.jpg',
 'cloudy14.jpg',
 'cloudy140.jpg',
 'cloudy141.jpg',
 'cloudy142.jpg',
 'cloudy143.jpg',
 'cloudy144.jpg',
 'cloudy145.jpg',
 'cloudy146.jpg',
 'cloudy147.jpg',
 'cloudy148.jpg

In [12]:
len(filenames)

1130

In [13]:
species = ['cloudy', 'rain', 'shine', 'sunrise']

In [14]:
# 创建train和test目录
if not os.path.exists(train_dir):
    os.mkdir(train_dir)

if not os.path.exists(test_dir):
    os.mkdir(test_dir)

In [16]:
# 分别在train和test目录下创建4中类别的目录
for train_or_test in ['train', 'test']:
    for spec in species:
        path = os.path.join(base_dir, train_or_test, spec)
        os.mkdir(path)

In [17]:
# python中的自带拷贝工具
import shutil

In [20]:
# 把dataset中的图片全部拷贝到train和test目录下的4个子目录中
for i, img in enumerate(filenames):
    for spec in species:
        if spec in img:
            img_path = os.path.join(base_dir, img)
            if i % 5 == 0:
                path = os.path.join(base_dir, 'test', spec, img)
            else:
                path = os.path.join(base_dir, 'train', spec, img)
            shutil.copy(img_path, path)

In [21]:
# 打印每个类别训练数据和测试数据分别有多少照片
for train_or_test in ['train', 'test']:
    for spec in species:
        print(train_or_test, spec, len(os.listdir(os.path.join(base_dir, train_or_test, spec))))

train cloudy 240
train rain 172
train shine 202
train sunrise 286
test cloudy 60
test rain 43
test shine 51
test sunrise 71


In [22]:
from torchvision import transforms

In [25]:
transform = transforms.Compose([
    # 统一缩放到96*96
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    # 正则化
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [26]:
import torchvision

In [28]:
train_ds = torchvision.datasets.ImageFolder(train_dir, transform=transform)

In [31]:
train_ds.classes

['cloudy', 'rain', 'shine', 'sunrise']

In [30]:
test_ds = torchvision.datasets.ImageFolder(test_dir, transform=transform)

In [32]:
train_ds.class_to_idx

{'cloudy': 0, 'rain': 1, 'shine': 2, 'sunrise': 3}

In [33]:
batch_size = 32

In [123]:
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, drop_last=True)

In [36]:
img, labels = next(iter(train_dl))

In [38]:
img.shape

torch.Size([32, 3, 96, 96])

In [39]:
labels

tensor([1, 2, 2, 3, 3, 1, 1, 1, 2, 2, 3, 0, 1, 0, 0, 1, 1, 0, 3, 0, 2, 2, 1, 3,
        2, 0, 3, 1, 1, 0, 3, 3])

In [41]:
img = img[0]

In [42]:
img.shape

torch.Size([3, 96, 96])

In [44]:
# torch.transpose()

In [45]:
img.min()

tensor(-0.9216)

In [46]:
img.max()

tensor(0.8902)

In [47]:
img = img + 1

In [48]:
img.min()

tensor(0.0784)

In [49]:
img.max()

tensor(1.8902)

In [50]:
img = img/2

In [51]:
img.max()

tensor(0.9451)

In [53]:
# 把图片的值控制到0-1 或者 0-255，就可以画图

In [92]:
# 定义模型
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)   # 16, 94,94
        self.pool = nn.MaxPool2d(2, 2)      # 16, 47, 47
        self.conv2 = nn.Conv2d(16, 32, 3)   # 32, 45,45 -> 32, 22, 22
        self.conv3 = nn.Conv2d(32, 64, 3)   # 63, 20, 20 -> 63, 10 ,10
        self.dropout = nn.Dropout()   # 遗忘层，可以把低于每个阈值的所有特征归零

        self.fc1 = nn.Linear(64 * 10 * 10, 1024)
        self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # x = x.view(-1, 64 * 10 *10)
        x = nn.Flatten()(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [93]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [94]:
model = Net()
if torch.cuda.is_available():
    model.to(device)

In [95]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [96]:
def fit(epoch, model, train_loader, test_loader):
    correct = 0
    total = 0
    running_loss = 0
    
    for x, y in train_loader:
        # 把数据放到GPU上去. 
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            y_pred = torch.argmax(y_pred, dim=1)
            correct += (y_pred == y).sum().item()
            total += y.size(0)
            running_loss += loss.item()
            
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
        
    # 测试过程
    test_correct = 0
    test_total = 0
    test_running_loss = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            y_pred = torch.argmax(y_pred, dim=1)
            test_correct += (y_pred == y).sum().item()
            test_total += y.size(0)
            test_running_loss += loss.item()
    test_epoch_loss = test_running_loss / len(test_loader.dataset)
    test_epoch_acc = test_correct / test_total

    print('epoch: ', epoch,
         'loss: ', round(epoch_loss, 3),
         'accuracy: ', round(epoch_acc, 3),
         'test_loss: ', round(test_epoch_loss, 3),
         'test_accuracy: ', round(test_epoch_acc, 3))
    return epoch_loss, epoch_acc, test_epoch_loss, test_epoch_acc

In [97]:
epochs = 20
train_loss = []
train_acc = []
test_loss = []
test_acc = []
for epoch in range(epochs):
    epoch_loss, epoch_acc, test_epoch_loss, test_epoch_acc = fit(epoch, model, train_dl, test_dl)
    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)
    
    test_loss.append(epoch_loss)
    test_acc.append(epoch_acc)

epoch:  0 loss:  0.028 accuracy:  0.594 test_loss:  0.021 test_accuracy:  0.756
epoch:  1 loss:  0.019 accuracy:  0.739 test_loss:  0.02 test_accuracy:  0.787
epoch:  2 loss:  0.016 accuracy:  0.806 test_loss:  0.021 test_accuracy:  0.782
epoch:  3 loss:  0.015 accuracy:  0.822 test_loss:  0.017 test_accuracy:  0.791
epoch:  4 loss:  0.012 accuracy:  0.858 test_loss:  0.019 test_accuracy:  0.827
epoch:  5 loss:  0.011 accuracy:  0.873 test_loss:  0.017 test_accuracy:  0.809
epoch:  6 loss:  0.01 accuracy:  0.898 test_loss:  0.02 test_accuracy:  0.804
epoch:  7 loss:  0.013 accuracy:  0.879 test_loss:  0.019 test_accuracy:  0.853
epoch:  8 loss:  0.01 accuracy:  0.872 test_loss:  0.017 test_accuracy:  0.853
epoch:  9 loss:  0.008 accuracy:  0.907 test_loss:  0.018 test_accuracy:  0.884
epoch:  10 loss:  0.008 accuracy:  0.911 test_loss:  0.014 test_accuracy:  0.88
epoch:  11 loss:  0.006 accuracy:  0.928 test_loss:  0.018 test_accuracy:  0.907
epoch:  12 loss:  0.005 accuracy:  0.944 te

In [131]:
### 添加BN层 batch_normallize
# 定义模型
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)   # 16, 94,94
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)      # 16, 47, 47
        self.conv2 = nn.Conv2d(16, 32, 3)   # 32, 45,45 -> 32, 22, 22
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3)   # 63, 20, 20 -> 63, 10 ,10
        self.bn3 = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout(0.5)   # 遗忘层，可以把低于每个阈值的所有特征归零

        self.fc1 = nn.Linear(64 * 10 * 10, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024)
        self.fc2 = nn.Linear(1024, 256)
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.bn1(x)
        x = self.pool(F.relu(self.conv2(x)))
        x = self.bn2(x)
        x = self.pool(F.relu(self.conv3(x)))
        x = self.bn3(x)

        # x = x.view(-1, 64 * 10 *10)
        x = nn.Flatten()(x)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.bn_fc1(x)

        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.bn_fc2(x)
        
        x = self.fc3(x)
        return x

In [132]:
model = Net()
if torch.cuda.is_available():
    model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [133]:
epochs = 10
train_loss = []
train_acc = []
test_loss = []
test_acc = []
for epoch in range(epochs):
    epoch_loss, epoch_acc, test_epoch_loss, test_epoch_acc = fit(epoch, model, train_dl, test_dl)
    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)
    
    test_loss.append(epoch_loss)
    test_acc.append(epoch_acc)

epoch:  0 loss:  0.015 accuracy:  0.81 test_loss:  0.102 test_accuracy:  0.393
epoch:  1 loss:  0.011 accuracy:  0.857 test_loss:  0.101 test_accuracy:  0.388
epoch:  2 loss:  0.008 accuracy:  0.906 test_loss:  0.101 test_accuracy:  0.455
epoch:  3 loss:  0.005 accuracy:  0.941 test_loss:  0.111 test_accuracy:  0.384
epoch:  4 loss:  0.006 accuracy:  0.922 test_loss:  0.116 test_accuracy:  0.406
epoch:  5 loss:  0.004 accuracy:  0.949 test_loss:  0.125 test_accuracy:  0.402
epoch:  6 loss:  0.004 accuracy:  0.955 test_loss:  0.121 test_accuracy:  0.415
epoch:  7 loss:  0.003 accuracy:  0.971 test_loss:  0.124 test_accuracy:  0.393
epoch:  8 loss:  0.002 accuracy:  0.982 test_loss:  0.134 test_accuracy:  0.388
epoch:  9 loss:  0.003 accuracy:  0.973 test_loss:  0.139 test_accuracy:  0.357
